# Bölüm 2 · Betimsel İstatistik Atölyesi

**GitHub + Google Colab pilotu · 27 Ekim 2026**

[Atölye sayfası](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b02/) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b02/GOREVLER.md) · [Veri/kaynak](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b02/VERI.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b02/COZUMLER.md)

**Başlangıç:** Colab'da Google hesabınızla giriş yapın. Düzenlemelerinizi saklamak için **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU çalışma zamanı yeterlidir; GPU gerekmez. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk çalıştırmada internet gerekir. Analiz verileri sabit bir GitHub sürümünden alınır; hesabınıza veya Drive'ınıza kodla erişilmez.

Bu defter: merkezi eğilim ve yayılımı hesaplama, dağılım ve eşleşmiş değişimleri görme, sonuçları yorumlama ve çıktı indirme. Çalışma bilgisayarda Jupyter ile de yapılabilir. Paket sürümleri aşağıda kaydedilir; Colab'ın ortamı değişebileceğinden sayısal kontrol hücresi önemlidir.

## 1 · Ortamı hazırla

Colab'da genellikle hazır bulunan dört paketi kullanıyoruz. Eksik paket varsa yalnız o paket kurulur; mevcut paketler topluca güncellenmez. Kurulumdan sonra Colab yeniden başlatma isterse oturumu yeniden başlatıp baştan çalıştırın.

In [ ]:
import importlib.util, subprocess, sys, os
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".b02-matplotlib"))
packages = {"numpy":"numpy>=1.26,<3", "pandas":"pandas>=2.2,<4", "matplotlib":"matplotlib>=3.8,<4", "scipy":"scipy>=1.11,<2"}
missing = [spec for name, spec in packages.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.stats import trim_mean
import json, hashlib, io, urllib.request, zipfile
VERSIONS = {"Python":sys.version.split()[0], "NumPy":np.__version__, "pandas":pd.__version__, "Matplotlib":matplotlib.__version__, "SciPy":scipy.__version__}
print(VERSIONS)
plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":11, "pdf.fonttype":42})

## 2 · Sabit sürümdeki veriyi yükle ve denetle

Kaynak: UCI Student Performance, Portekizce dersi (Cortez & Silva, 2008; veri DOI: [10.24432/C5TG7T](https://doi.org/10.24432/C5TG7T), CC BY 4.0). Buradaki dosya **kaynak sırasındaki ilk 80 GP okul kaydının G1/G3 izdüşümüdür**; 80 yeni katılımcı toplanmamıştır ve bütün öğrencileri temsil eden rastgele örneklem değildir. Yerel sütun `kaynak_satir` bu aktarımın sıra kimliğidir. G1 ilk dönem, G3 son nottur; puan aralığı 0–20'dir. Aynı satır aynı öğrenciyi temsil eder.

Kod, değişmeyen Git commit'inden CSV alır ve SHA256 ile doğrular. Önceki çalıştırmadan kalan dosya varsa o da aynı denetimden geçer. Hata alırsanız ağ bağlantısını ve dosyayı kontrol edin; beklenen hash'i değiştirmeyin.

In [ ]:
DATA_COMMIT = "047d96c3d18ef77bc127ca5b106f6a64d3083e79"
DATA_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/uygulamalar/b12-regresyon/kaynak/uci-portekizce-ilk80-notlar.csv"
EXPECTED_SHA256 = "cff1ee5aaa3058bd48db858789a46a24d12c7ac072b1f5f1c3171dd2a1995a55"
work = Path.cwd() / "b02-atolye"
work.mkdir(exist_ok=True)
source = work / "kaynak-notlar.csv"
if source.exists():
    raw = source.read_bytes()
else:
    try:
        with urllib.request.urlopen(DATA_URL, timeout=30) as response:
            raw = response.read()
    except Exception as exc:
        raise RuntimeError("CSV indirilemedi. İnternet bağlantısını kontrol edip bu hücreyi yeniden çalıştırın. Kaynak: " + DATA_URL) from exc
if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
    raise ValueError("Kaynak dosyanın SHA256 değeri farklı. Dosyayı doğrulayın; hash denetimini kaldırmayın.")
source.write_bytes(raw)
data = pd.read_csv(io.BytesIO(raw))
assert list(data.columns) == ["kaynak_satir", "school", "G1", "G3"]
assert len(data) == 80 and not data.isna().any().any()
assert data.school.eq("GP").all() and data.kaynak_satir.tolist() == list(range(1,81))
assert data[["G1","G3"]].ge(0).all().all() and data[["G1","G3"]].le(20).all().all()
# Ana kaynak korunur; fark ayrı çalışma kopyasına eklenir.
df = data.assign(fark_G3_G1=data.G3-data.G1)
print("80 kayıt doğrulandı. Veri commit:", DATA_COMMIT)
print(df.head(8).to_string(index=False))

## 3 · Hesaplamadan önce tahmin et

- G1'deki sıfır puan ortalama ve medyanı aynı ölçüde etkiler mi?
- Son notların ortalaması yükselse bile her öğrenci ilerlemiş olabilir mi?
- `ddof=1` ile hesaplanan standart sapma ile ortalamanın standart hatası aynı şeyi mi anlatır?

Yanıtlarınızı aşağıya yazın. Bu hücreyi doldurmak hesapları durdurmaz.

In [ ]:
# Kendi yanıtlarınızı yazın; bu bir not alma alanıdır.
yanitlarim = {
    "ortalama_medyan": "",
    "her_ogrenci_ilerledi_mi": "",
    "SS_ve_SH": ""
}

## 4 · Merkezi eğilim ve yayılım

Standart sapmada örneklem böleni **n−1**; çeyreklerde **NumPy linear / R tip 7** kullanılır. Ham MAD medyandan mutlak sapmaların medyanıdır; ölçeklenmiş MAD değildir. Kırpılmış ortalama her uçtan %10 kayıt çıkarır ve başka bir özet hedefidir. Aşağıdaki tabloya standart hata eklemiyoruz: standart hata, kayıtların yayılımını betimleyen SS'nin yerine geçmez.

In [ ]:
def ozet(x):
    x = np.asarray(x, dtype=float)
    q1, medyan, q3 = np.quantile(x, [.25,.5,.75], method="linear")
    return {"n":len(x), "Ortalama":x.mean(), "Medyan":medyan,
            "SS (n-1)":x.std(ddof=1), "Q1 (tip 7)":q1, "Q3 (tip 7)":q3,
            "IQR":q3-q1, "Ham MAD":np.median(np.abs(x-medyan)),
            "%10 kırpılmış ortalama":trim_mean(x,.1), "Min":x.min(), "Maks":x.max()}
tablo = pd.DataFrame({name:ozet(df[name]) for name in ["G1","G3","fark_G3_G1"]}).T
print(tablo.round(4).to_string())

## 5 · Notların dağılımını gör

Üst panel her tam puandaki kayıt sayısını, alt panel `F(x)=P̂(X≤x)` ampirik dağılımını gösterir. Alt panelde bir nottaki sıçramanın yüksekliği o puanın göreli sıklığıdır. Renkler yanında farklı çizgi ve işaretler de kullanılır.

**Soru:** Ortalama ve medyan yakınsa dağılımın simetrik olduğu sonucuna tek başına varabilir misiniz? Tabloyu ve bütün grafiği birlikte yorumlayın.

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(8,7),layout="constrained")
for name,color,marker,style in [("G1","#0072B2","o","-"),("G3","#D55E00","s","--")]:
    freq = df[name].value_counts().reindex(range(21),fill_value=0)
    axes[0].plot(freq.index,freq.values,color=color,marker=marker,ls=style,label=name)
    grid = np.arange(-1,22)
    ecdf = np.array([(df[name]<=x).mean() for x in grid])
    axes[1].step(grid,ecdf,where="post",color=color,ls=style,label=name)
axes[0].set(xlabel="Not (0–20)",ylabel="Kayıt sayısı",title="Tam puan frekansları (çizgiler görsel kılavuzdur)")
axes[1].set(xlabel="Not eşiği x",ylabel="Notu x veya altında olan oran",xlim=(-.5,20.5),ylim=(0,1.02),title="Ampirik dağılım: F(x) = oran(not ≤ x)")
for ax in axes: ax.legend();ax.grid(alpha=.2)
fig.savefig(work/"dagilim.pdf",bbox_inches="tight")
plt.show()

## 6 · Aynı öğrencinin değişimini izle

Soldaki çizgiler **aynı satırın** iki notunu bağlar; kalın kesikli çizgi yalnız grup ortalamalarını bağlar. Sağda her öğrenci için `G3−G1` dağılımı vardır. Notlar arasındaki zaman farkı tek başına bir öğretim yönteminin nedensel etkisini göstermez.

**Soru:** Sadece iki grup ortalamasını göstermek, azalan ve değişmeyen notlar hakkında neyi gizler?

In [ ]:
delta = df.fark_G3_G1.to_numpy()
sayim = {"Artan":int((delta>0).sum()), "Aynı":int((delta==0).sum()), "Azalan":int((delta<0).sum())}
print(sayim)
fig, axes = plt.subplots(1,2,figsize=(10,4),layout="constrained")
for row in df.itertuples(): axes[0].plot([0,1],[row.G1,row.G3],color="#555555",alpha=.2,lw=.8)
axes[0].plot([0,1],[df.G1.mean(),df.G3.mean()],color="#0072B2",marker="o",ls="--",lw=3,label="Ortalamalar")
axes[0].set(xticks=[0,1],xticklabels=["G1","G3"],ylabel="Not (0–20)",ylim=(0,20),title="Eşleşmiş öğrenci notları");axes[0].legend()
axes[1].hist(delta,bins=np.arange(delta.min()-.5,delta.max()+1.5),color="#0072B2",edgecolor="white")
axes[1].axvline(0,color="black",ls="--");axes[1].set(xlabel="G3 − G1 (puan)",ylabel="Kayıt sayısı",title="Kişi içi farkların dağılımı")
fig.savefig(work/"eslesmis-degisim.pdf",bbox_inches="tight")
plt.show()

## 7 · Bir kaydın etkisini ayrı duyarlılık analiziyle incele

Kaynak satır 1'i geçici olarak dışarıda bırakıp özetleri karşılaştırın. Bu **otomatik veri temizleme veya hata ilanı değildir**; ana veri ve ana sonuç korunur. Sıfır notun nedeni bu izdüşüm dosyasıyla bilinemez. Bir kaydın etkili olması o kaydın hatalı olduğunu kanıtlamaz.

In [ ]:
duyarlilik = df.loc[df.kaynak_satir.ne(1)]
karsilastirma = pd.DataFrame({"Ana 80 kayıt":ozet(df.G1), "Satır 1 hariç 79":ozet(duyarlilik.G1)})
print(karsilastirma.loc[["n","Ortalama","Medyan","SS (n-1)"]].round(4).to_string())
assert len(df)==80 and source.read_bytes()==raw

## 8 · Sayısal kontrol

Bu hücre temel sonuçları kitabın sürümlenmiş referanslarıyla karşılaştırır. Bu bir bilimsel varsayım testi veya SPSS doğrulaması değildir. Hata alırsanız önce veri, kod ve paket sürümlerini inceleyin; referansları sonuca uydurmayın.

In [ ]:
checks = {
    "G1 ortalama":bool(np.isclose(df.G1.mean(),12.175)),
    "G3 ortalama":bool(np.isclose(df.G3.mean(),12.6375)),
    "G1 örneklem SS":bool(np.isclose(df.G1.std(ddof=1),2.4275580270476635)),
    "G3 örneklem SS":bool(np.isclose(df.G3.std(ddof=1),2.026681205533016)),
    "G1 tip 7 çeyrekler":bool(np.allclose(np.quantile(df.G1,[.25,.75],method="linear"),[11,13.25])),
    "Ortalama eşleşmiş fark":bool(np.isclose(delta.mean(),.4625)),
    "Medyan eşleşmiş fark":bool(np.isclose(np.median(delta),0)),
    "Değişim sayımları":sayim=={"Artan":32,"Aynı":31,"Azalan":17},
    "Ana kaynak korunmuş":source.read_bytes()==raw
}
assert all(checks.values()),checks
print("9 sayısal/veri kontrolü geçti.")
print(checks)

## 9 · Kısa rapor ve çıktı paketi

**Siz tamamlayın:** Örneklemin kaynağını ve sınırlılığını yazın; G1/G3 için ortalama, medyan ve SS'yi raporlayın. Artan/aynı/azalan kayıtları belirtin. Ana ve duyarlılık sonuçlarını ayırın. Betimsel bulgudan nedensel etki çıkarmayın.

Son hücre özet CSV'lerini, iki PDF grafiğini, kendi yanıtlarınızı, kaynak/sürüm kaydını ve kontrolleri tek ZIP'e toplar. **Colab'da soldaki Dosyalar panelini yenileyip `b02-atolye-ciktilar.zip` dosyasının menüsünden İndir'i seçin.** Çalışma zamanı kapanmadan dosyaları indirin veya defteri Drive'a kaydedin; geçici dosyalar kalıcı değildir. SPSS dosyaları Colab'da yürütülmez.

In [ ]:
tablo.to_csv(work/"betimsel-ozet.csv",encoding="utf-8-sig")
karsilastirma.to_csv(work/"duyarlilik.csv",encoding="utf-8-sig")
manifest = {"data_commit":DATA_COMMIT,"source_url":DATA_URL,"sha256":EXPECTED_SHA256,
            "versions":VERSIONS,"checks":checks,"paired_counts":sayim,"answers":yanitlarim,
            "quantiles":"NumPy linear / R type 7", "sd_ddof":1,
            "scope":"Kaynak sırasındaki 80 GP kaydı; betimsel analiz, nedensel etki veya SPSS yürütmesi değil"}
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
archive = Path.cwd()/"b02-atolye-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for name in ["betimsel-ozet.csv","duyarlilik.csv","dagilim.pdf","eslesmis-degisim.pdf","calisma-kaydi.json"]:
        z.write(work/name,name)
print("İndirmeye hazır:",archive.name)